In [ ]:
import sys; sys.path.append("..")           # so `src` imports work from /notebooks
import numpy as np, pandas as pd
from src.features.extractor import extract_frame, FEATURES

df = pd.read_csv("../data/processed/messages.csv")
df = df.sort_values(["conversation_id", "turn_index"]).reset_index(drop=True)
df = extract_frame(df)
df.to_csv("../data/processed/message_features.csv", index=False)
df[["message", *FEATURES]].sample(8, random_state=0)

def audit(g):
    X = g[FEATURES]
    return pd.DataFrame({"mean": X.mean(), "std": X.std(), "pct_zero": (X == 0).mean()})

audit_by_ds = pd.concat({k: audit(g) for k, g in df.groupby("dataset")}, axis=1)
audit_by_ds.round(3)

def icc1(x, g):
    d = pd.DataFrame({"x": x, "g": g}).dropna()
    grp = d.groupby("g")["x"]
    k, n, m = grp.size(), len(d), grp.ngroups
    if m < 2 or n <= m: return np.nan
    ssb = (k * (grp.mean() - d.x.mean()) ** 2).sum()
    ssw = ((d.x - grp.transform("mean")) ** 2).sum()
    msb, msw = ssb / (m - 1), ssw / (n - m)
    k0 = (n - (k ** 2).sum() / n) / (m - 1)
    den = msb + (k0 - 1) * msw
    return (msb - msw) / den if den > 0 else np.nan

df["spk_key"] = df["conversation_id"].astype(str) + "_" + df["speaker_id"].astype(str)
KEY = ["log_words", "avg_sent_len", "punct_density", "n_questions",
       "n_exclam", "caps_ratio", "starts_lower", "has_emoji"]
icc = pd.DataFrame({ds: {f: icc1(g[f], g["spk_key"]) for f in KEY}
                    for ds, g in df.groupby("dataset")})
icc.round(3)

df["prev_log_words"] = df.groupby("conversation_id")["log_words"].shift(1)
conf = {ds: g[["log_words", "prev_log_words"]].corr(method="spearman").iloc[0, 1]
        for ds, g in df.dropna(subset=["prev_log_words"]).groupby("dataset")}
pd.Series(conf, name="spearman(reply_len, prev_len)").round(3)
